In [ ]:
import math
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

sys.path.append('.')
from harness.measurement import ReconciledMeasurementHarness
from methods.pinn import create_heat_pinn_factory
from methods.fno import create_fno_factory

def get_exact_solution(k):
    x_test = torch.linspace(0, 1, 100).view(-1, 1)
    decay = math.exp(-0.01 * (k * math.pi)**2 * 0.01)
    exact_u = torch.sin(k * math.pi * x_test) * decay
    return exact_u

if __name__ == "__main__":
    torch.manual_seed(42)
    harness = ReconciledMeasurementHarness()

    print("=== PREPARING SOLVERS ===")
    my_pinn_factory = create_heat_pinn_factory(alpha=0.01, epochs=500)
    my_fno_factory = create_fno_factory(epochs=100, num_samples=2000)

    print("\n=== RUNNING BENCHMARKS ===")
    results_pinn = harness.benchmark_pinn_solver(
        name="Standard PINN",
        pinn_solver_factory=my_pinn_factory,
        u_exact_fn=get_exact_solution,
        sample_instances=[1, 2],  
        n_runs=2
    )

    results_fno = harness.benchmark_pinn_solver( 
        name="Standard FNO",
        pinn_solver_factory=my_fno_factory, 
        u_exact_fn=get_exact_solution,
        sample_instances=[1, 2], 
        n_runs=2
    )

    pinn_train_cost = results_pinn['train_time_sec'] 
    pinn_instance_cost = results_pinn['per_instance_sec']
    _, fno_info = my_fno_factory(1) 
    fno_train_cost = fno_info['amortized_train_time_sec']
    fno_instance_cost = results_fno['per_instance_sec'] 

    crossover_N = (fno_train_cost - pinn_train_cost) / (pinn_instance_cost - fno_instance_cost)

    print("=== Computational Crossover Analysis ===")
    print(f"PINN Cost per new instance: {pinn_instance_cost:.4f} sec")
    print(f"FNO Cost per new instance:  {fno_instance_cost:.6f} sec")
    print(f"Break-even point: The FNO becomes cheaper after exactly {crossover_N:.2f} instances.")

    N_max = int(max(crossover_N * 2, 10)) 
    N_values = np.arange(1, N_max + 1)
    pinn_total_costs = pinn_train_cost + (N_values * pinn_instance_cost)
    fno_total_costs = fno_train_cost + (N_values * fno_instance_cost)

    os.makedirs('figures', exist_ok=True)
    plt.figure(figsize=(10, 6))
    plt.plot(N_values, pinn_total_costs, label='PINN (Instance-Based)', color='red', linewidth=2)
    plt.plot(N_values, fno_total_costs, label='FNO (Amortized Operator)', color='blue', linewidth=2)
    if crossover_N > 0:
        plt.axvline(x=crossover_N, color='gray', linestyle='--', label=f'Crossover at N≈{int(crossover_N)}')
        plt.plot(crossover_N, fno_train_cost + crossover_N * fno_instance_cost, 'ko') 

    plt.title("Total Computational Cost to Handle N PDE Instances", fontsize=14)
    plt.xlabel("Number of Unique Problem Instances (N)", fontsize=12)
    plt.ylabel("Total Compute Time (Seconds)", fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('figures/amortization_analysis.png')